In [1]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), incompatible header (1), mismatched flags (5))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), incompatible header (2), mismatched flags (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
structured_box_mesh(n=40)

mat = Material("body")
P = Problem([mat], type=:ScalarField, dim=3, field=:p)

Problem("structured_box", :ScalarField, 3, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 68921, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :rhs)

In [3]:
f(x, y, z) = 1 + sin(10x) + sin(10y) + sin(10z)
s = scalarField(P, "body", f)
S = ScalarField(P, "body", f)

elementwise ScalarField
[[1.494807918509046; 1.247403959254523; … ; 1.247403959254523; 1.494807918509046;;], [1.742211877763569; 1.494807918509046; … ; 1.479425538604203; 1.726829497858726;;], [1.9742334571132492; 1.726829497858726; … ; 1.6816387600233342; 1.9290427192778572;;], [2.17644667853238; 1.929042719277857; … ; 1.8414709848078965; 2.0888749440624195;;], [2.336278903316943; 2.0888749440624195; … ; 1.9489846193555862; 2.196388578610109;;], [2.4437925378646326; 2.196388578610109; … ; 1.9974949866040546; 2.2448989458585773;;], [2.492302905113101; 2.2448989458585773; … ; 1.9839859468739367; 2.23138990612846;;], [2.478793865382983; 2.2313899061284594; … ; 1.9092974268256817; 2.1567013860802047;;], [2.404105345334728; 2.1567013860802047; … ; 1.778073196887921; 2.025477156142444;;], [2.2728811153969675; 2.025477156142444; … ; 1.5984721441039564; 1.8458761033584794;;]  …  [0.8499577549959993; 1.0744596722630955; … ; 1.355560391866629; 1.1310584745995327;;], [0.9065565573324366; 1.13105

In [4]:
GC.gc()
@time ∫(P, "body", f)

  0.200511 seconds (41 allocations: 42.490 MiB)


1.551721167137964

In [5]:
GC.gc()
@time ∫(P, "body", s)

  0.707575 seconds (482.07 k allocations: 65.175 MiB, 73.72% compilation time)


1.548844911678299

In [6]:
GC.gc()
@time ∫(P, "body", S)

  0.210103 seconds (45 allocations: 42.483 MiB)


1.548844911678299

In [7]:
dim, tag = only(gmsh.model.getEntitiesForPhysicalName("body"))

elementTypes, elementTags, elemNodeTags =
    gmsh.model.mesh.getElements(dim, tag)

et = only(elementTypes)

elementName, dim, order1, numNodes,
localNodeCoord, numPrimaryNodes =
    gmsh.model.mesh.getElementProperties(et)

intPoints, intWeights =
    gmsh.model.mesh.getIntegrationPoints(et, "Gauss$order1")

GC.gc()
@time jacAll, detAll, coordAll =
    gmsh.model.mesh.getJacobians(et, intPoints, tag);

  0.158392 seconds (145 allocations: 38.093 MiB, 13.15% compilation time)


In [8]:
@time G_old = ∇_old(S)
@time G_new = ∇(S)

  8.798702 seconds (32.71 M allocations: 2.435 GiB, 5.39% gc time, 52.06% compilation time)
  0.979079 seconds (664.58 k allocations: 101.009 MiB, 1.64% gc time, 70.04% compilation time)


elementwise VectorField
[[9.896158370180913; 9.89615837018092; … ; 9.89615837018092; 9.896158370180927;;], [9.280863173987186; 9.896158370180926; … ; 9.89615837018092; 9.896158370180927;;], [8.088528856765237; 9.896158370180913; … ; 9.896158370180927; 9.896158370180899;;], [6.393288991382519; 9.896158370180906; … ; 9.896158370180927; 9.896158370180927;;], [4.300545381907597; 9.896158370180913; … ; 9.896158370180899; 9.896158370180956;;], [1.9404146899387342; 9.89615837018094; … ; 9.896158370180899; 9.896158370180942;;], [-0.5403615892047071; 9.896158370180938; … ; 9.896158370180927; 9.896158370180913;;], [-2.9875408019302085; 9.89615837018094; … ; 9.896158370180913; 9.896158370180913;;], [-5.248969197510434; 9.896158370180919; … ; 9.896158370180899; 9.896158370180942;;], [-7.184042111358593; 9.896158370180936; … ; 9.896158370180913; 9.896158370180913;;]  …  [2.2639520934574904; -8.98007669068388; … ; -8.98007669068388; -8.98007669068388;;], [-0.20962129951176678; -8.98007669068388; … ;

In [9]:
G_old.numElem == G_new.numElem

true

In [10]:
maximum(
    maximum(abs, G_old.A[i] .- G_new.A[i])
    for i in eachindex(G_old.A)
)

1.9895196601282805e-13

In [11]:
G_new = ∇(S)  # compilation

GC.gc()
@time G_new = ∇(S)

  0.282127 seconds (128.06 k allocations: 74.893 MiB)


elementwise VectorField
[[9.896158370180913; 9.89615837018092; … ; 9.89615837018092; 9.896158370180927;;], [9.280863173987186; 9.896158370180926; … ; 9.89615837018092; 9.896158370180927;;], [8.088528856765237; 9.896158370180913; … ; 9.896158370180927; 9.896158370180899;;], [6.393288991382519; 9.896158370180906; … ; 9.896158370180927; 9.896158370180927;;], [4.300545381907597; 9.896158370180913; … ; 9.896158370180899; 9.896158370180956;;], [1.9404146899387342; 9.89615837018094; … ; 9.896158370180899; 9.896158370180942;;], [-0.5403615892047071; 9.896158370180938; … ; 9.896158370180927; 9.896158370180913;;], [-2.9875408019302085; 9.89615837018094; … ; 9.896158370180913; 9.896158370180913;;], [-5.248969197510434; 9.896158370180919; … ; 9.896158370180899; 9.896158370180942;;], [-7.184042111358593; 9.896158370180936; … ; 9.896158370180913; 9.896158370180913;;]  …  [2.2639520934574904; -8.98007669068388; … ; -8.98007669068388; -8.98007669068388;;], [-0.20962129951176678; -8.98007669068388; … ;

In [12]:
GC.gc()
@time G_old = ∇_old(S)

  5.257223 seconds (25.79 M allocations: 2.106 GiB, 5.39% gc time)


elementwise VectorField
[[9.89615837018092; 9.896158370180919; … ; 9.89615837018092; 9.896158370180927;;], [9.2808631739872; 9.896158370180926; … ; 9.89615837018092; 9.896158370180899;;], [8.088528856765251; 9.896158370180906; … ; 9.896158370180927; 9.896158370180913;;], [6.393288991382519; 9.896158370180897; … ; 9.896158370180927; 9.896158370180942;;], [4.300545381907611; 9.896158370180915; … ; 9.896158370180913; 9.896158370180942;;], [1.9404146899387342; 9.896158370180961; … ; 9.896158370180899; 9.896158370180927;;], [-0.5403615892047355; 9.896158370180938; … ; 9.896158370180927; 9.896158370180913;;], [-2.9875408019301943; 9.896158370180926; … ; 9.896158370180913; 9.896158370180942;;], [-5.248969197510391; 9.896158370180936; … ; 9.896158370180913; 9.896158370180927;;], [-7.184042111358579; 9.896158370180924; … ; 9.896158370180935; 9.896158370180927;;]  …  [2.2639520934574904; -8.98007669068388; … ; -8.980076690683859; -8.98007669068388;;], [-0.20962129951177388; -8.98007669068388; … 

In [13]:
S2 = nodesToElements(s, onPhysicalGroup="right")

dS2 = ∇(S2)

elementwise VectorField
[[0.0; 9.896158370180917; … ; 9.896158370180913; 9.896158370180917;;], [0.0; 9.2808631739872; … ; 9.2808631739872; 9.896158370180913;;], [0.0; 8.088528856765244; … ; 8.088528856765244; 9.896158370180913;;], [0.0; 6.393288991382491; … ; 6.393288991382484; 9.896158370180913;;], [0.0; 4.30054538190759; … ; 4.30054538190759; 9.896158370180913;;], [0.0; 1.9404146899387271; … ; 1.9404146899387342; 9.896158370180927;;], [0.0; -0.5403615892047; … ; -0.5403615892046929; 9.89615837018092;;], [0.0; -2.9875408019302085; … ; -2.9875408019302085; 9.896158370180913;;], [0.0; -5.248969197510419; … ; -5.248969197510419; 9.896158370180913;;], [0.0; -7.1840421113586; … ; -7.184042111358593; 9.896158370180906;;]  …  [0.0; 2.2639520934574904; … ; 2.2639520934574904; -8.98007669068388;;], [0.0; -0.209621299511781; … ; -0.209621299511781; -8.98007669068388;;], [0.0; -2.670161455361658; … ; -2.670161455361658; -8.980076690683873;;], [0.0; -4.9646839046339934; … ; -4.96468390463399; -8.

In [14]:
dS2_old = ∇_old(S2)

elementwise VectorField
[[0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;]  …  [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0; 0.0; … ; 0.0; 0.0;;], [-25.282533306724464; 0.0; … ; -8.980076690683877; -8.980076690683877;;]]

In [15]:
showElementResults(dS2)
showElementResults(dS2_old)

1

In [16]:
S2 = nodesToElements(s, onPhysicalGroup="right")
dS2 = ∇(S2)

@show S2.numElem == dS2.numElem
@show length(S2.numElem)
@show length(dS2.numElem)

showElementResults(dS2)

S2.numElem == dS2.numElem = true
length(S2.numElem) = 1600
length(dS2.numElem) = 1600


2

In [17]:
maximum(
    maximum(abs, e[1:3:end, :])
    for e in dS2.A
)

0.0

In [18]:
@time nodesToElements(S)
@time nodesToElements(s)
@time elementsToNodes(s)
@time elementsToNodes(S)

  0.016566 seconds (34 allocations: 2.000 KiB, 99.92% compilation time)
  0.015856 seconds (128.04 k allocations: 16.265 MiB)
  0.000008 seconds (1 allocation: 64 bytes)
  0.022284 seconds (45 allocations: 7.950 MiB)


nodal ScalarField
[0.4559788891106302; 1.0; … ; 0.28581049229364514; 0.04144241913317903;;]

In [19]:
@time dSx = ∇(S)[1]
@time dSx = ∇(s)[1]
@time dSx_old = ∂x(S)
@time ∂x(s)

@time ∫(P, "body", (dSx - dSx_old)^2)

  0.700899 seconds (303.61 k allocations: 86.552 MiB, 1.76% gc time, 8.72% compilation time)
  0.396423 seconds (384.10 k allocations: 100.435 MiB, 14.67% gc time)
  0.481225 seconds (296.13 k allocations: 86.178 MiB, 32.59% gc time, 3.62% compilation time)
  0.297990 seconds (384.09 k allocations: 100.434 MiB)
  1.265305 seconds (991.12 k allocations: 107.060 MiB, 2.32% gc time, 79.66% compilation time)


0.0

In [20]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
